In [1]:
import pandas as pd

# Loading in data and renaming columns
def load_data(file_path):
    df = pd.read_csv(file_path, header=None, sep="\t")

    df = df.rename(columns={
        0: "id",
        1: "modified",
        2: "sequence",
        3: "position"
    })
    df["position"] = df["position"].astype("Int64")
    return df



file = "/home/callum/projects/sno_analysis/data/raw/SNO_combined.txt"
df = load_data(file)

print("duplicates...")
print(df.duplicated().sum())


print(df.dtypes)

# function exploring dataset and giving basic info and counts
# def explore_data(df):
#     """Print a summary of shape, dtypes, nulls, and key value counts."""
#     print("Shape:", df.shape)
#     print("\nDtypes:\n", df.dtypes)
#     print("\nMissing values per column:\n", df.isnull().sum())
#     print("\nUnique proteins:", df["id"].nunique())
#     print(f"\nClass balance (modified): {df['modified'].value_counts()}\n")
# explore_data(df)


# --- Identifying conflicting sites ---
# for each id and position (each unique site), how many unique modified values are there?
# ie are there both positives and negatives for the same site
site_counts = df.groupby(["id", "position"])["modified"].nunique()
# this gives a series with id and position as index and 1 column - the number of unique entries

# filter for values greater than 1 - ie 2 different modification values present for the same site
conflicting_sites = site_counts[site_counts > 1]
print(conflicting_sites)

# reset the index to make id and position columns in a proper dataframe
conflicting_sites_df = conflicting_sites.reset_index()

# rename the count column from this df to identify column that marks conflicts
conflicting_sites_df = conflicting_sites_df.rename(columns={"modified": "conflicts"})
print(conflicting_sites_df)


# --- Merging conflict info back onto the full dataset ---
print("merging dataframes...")
merged = pd.merge(df, conflicting_sites_df, on=["id", "position"], how="left", indicator=True)

# confirming number of matched rows
# there are 334 - conflicting pairs have an entry for both the 0 and the 1 modification
print(merged["_merge"].value_counts())


# --- Filtering: keep non-conflicting rows, plus only the positive row for conflicts ---
before = int(merged.shape[0])

condition1 = merged["_merge"] == "left_only"
condition2 = (merged["_merge"] == "both") & (merged["modified"] == 1)
merged_filtered = merged[condition1 | condition2]

after = int(merged_filtered.shape[0])
print(f"Rows dropped: {before - after}")

# drop the helper columns now that filtering is done
to_drop = ["_merge", "conflicts"]
merged_filtered_clean = merged_filtered.drop(to_drop, axis="columns")

print(merged_filtered_clean.head())
print(merged_filtered_clean.shape[0])



duplicates...
0
id            str
modified    int64
sequence      str
position    Int64
dtype: object
id      position
A2ASS6  13296       2
        14760       2
        18679       2
        21561       2
        21780       2
                   ..
Q9WUM4  456         2
Q9WV60  14          2
        317         2
Q9Z0E0  415         2
        725         2
Name: modified, Length: 167, dtype: int64
         id  position  conflicts
0    A2ASS6     13296          2
1    A2ASS6     14760          2
2    A2ASS6     18679          2
3    A2ASS6     21561          2
4    A2ASS6     21780          2
..      ...       ...        ...
162  Q9WUM4       456          2
163  Q9WV60        14          2
164  Q9WV60       317          2
165  Q9Z0E0       415          2
166  Q9Z0E0       725          2

[167 rows x 3 columns]
merging dataframes...
_merge
left_only     23897
both            334
right_only        0
Name: count, dtype: int64
Rows dropped: 167
       id  modified                         

# made into a function


In [19]:
import pandas as pd
df = pd.read_csv("/home/callum/projects/sno_analysis/data/processed/df_for_regression.csv")
# df = df.drop("Unnamed: 0", axis = "columns")
# df.to_csv("/home/callum/projects/sno_analysis/data/processed/df_for_regression.csv", index  = False)

sites = df[["id", "position","modified"]]
colnames = "protein_id,lysine_position,ub".split(",")
sites.columns = colnames
sites.to_csv("/home/callum/projects/sno_analysis/data/processed/sites.csv", index = False)

sites_test = sites.head(500)
sites_test.to_csv("/home/callum/projects/sno_analysis/data/processed/sites_test.csv", index = False)



In [2]:
def resolve_label_conflicts(df, group_cols, label_col, keep_value=1):
    """
    For rows sharing the same group_cols (e.g. id + position), where label_col
    has more than one distinct value, keep only the row where label_col == keep_value.
    Rows with no conflict are left untouched.
    """
    # count distinct label values per group
    label_counts = df.groupby(group_cols)[label_col].nunique()
    conflicting_groups = label_counts[label_counts > 1].reset_index()
    conflicting_groups = conflicting_groups.rename(columns={label_col: "n_unique_labels"})

    # flag which rows belong to a conflicting group
    merged = pd.merge(df, conflicting_groups, on=group_cols, how="left", indicator=True)

    # keep: non-conflicting rows, plus only the "keep_value" row within conflicting groups
    no_conflict = merged["_merge"] == "left_only"
    resolved_conflict = (merged["_merge"] == "both") & (merged[label_col] == keep_value)

    result = merged[no_conflict | resolved_conflict].drop(columns=["_merge", "n_unique_labels"])
    return result.reset_index(drop=True)

In [3]:
# filtering for sites

print(merged_filtered_clean.head())
positive_condition = merged_filtered_clean["modified"] == 1
negative_condition = merged_filtered_clean["modified"] == 0 

positive_sites = merged_filtered_clean[positive_condition]
negative_sites = merged_filtered_clean[negative_condition]

df = merged_filtered_clean
print(merged_filtered_clean.shape)
both = merged_filtered_clean.shape[0]
positive = positive_sites.shape[0]
negative = negative_sites.shape[0]

print(positive_sites.shape)
print(negative_sites.shape)

print(f"{positive + negative == both}")


positive_sites2 = merged_filtered_clean.query("modified == 1")
negative_sites2 = merged_filtered_clean.query("modified == 0")

       id  modified                                   sequence  position
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17
(24064, 4)
(3774, 4)
(20290, 4)
True


In [4]:
# counting / filtering for proteins based on how many sites they have - can identify highly modified proteins
series = df.groupby(["id"]).size()
print(series)

#make a dataframe
#series = series.reset_index()
print(series)

print(series.shape)

#filter based on 5 modifications 
mask = series >= 5
excluded_mask = series < 5
# apply the mask
series_filtered = series[mask]
series_filtered = series_filtered.reset_index()

excluded = series[excluded_mask]
excluded = excluded.reset_index()

# # check how many proteins appear 5 or more times

print(f"proteins > 5:{series_filtered.shape}")

# merge using array of protein ids 

df_merged = pd.merge(df, series_filtered, on = "id", how = "inner", indicator = False)

excluded_rows = pd.merge(df, excluded, on="id", how="inner")

print("================##===============")
print(f"proteins > 5:{series_filtered.shape}")
print(f"original df:{df.shape}")
print(f"new dataframe: {df_merged.shape}")
print(f"ecluded: {excluded.shape}")

print("================##===============")

# not sure how to sanity check

print(f"{df_merged.shape[0] + excluded_rows.shape[0] == df.shape[0]}")


id
A2A432     12
A2AAJ9    237
A2AJB2      3
A2AJI0      6
A2AN08    122
         ... 
Q9Z2W0     10
Q9Z2X1      6
Q9Z2Z6      6
Q9ZNZ7     25
Q9ZVA2     12
Length: 2217, dtype: int64
id
A2A432     12
A2AAJ9    237
A2AJB2      3
A2AJI0      6
A2AN08    122
         ... 
Q9Z2W0     10
Q9Z2X1      6
Q9Z2Z6      6
Q9ZNZ7     25
Q9ZVA2     12
Length: 2217, dtype: int64
(2217,)
proteins > 5:(1572, 2)
================##===============
proteins > 5:(1572, 2)
original df:(24064, 4)
new dataframe: (22316, 5)
ecluded: (645, 2)
================##===============
True


In [5]:
if 0 in df_merged.columns:
    df_merged = df_merged.drop(columns=[0])

print(df_merged.head())
df = df_merged



def get_top_n_proteins(df, n, colname):
    series = df.groupby(colname).size()
    
    # turn into df so we can use it 
    series = series.reset_index()
    series = series.rename(columns={0: "occurrences"})

    # now applying .nlargest - this sorts AND selects in one
    top = series.nlargest(n = n, columns = "occurrences")

    # merging dataframe to create master table of top 20 proteins
    top_merged = pd.merge(df, top, on = "id", how = "inner", indicator= False)

    # sanity check - should be 20 proteins
    print(f"number of unique proteins in final df: {top_merged["id"].nunique()}")
    
    return top_merged

print("###===###")
top_ps = get_top_n_proteins(df, 5, "id")
print(top_ps)



       id  modified                                   sequence  position
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17
###===###
number of unique proteins in final df: 5
          id  modified                                   sequence  position  \
0     A2ASS6         0  SEPSDPAVAGDPIYPPGPPSCPEVKDKTKSSISLAWKPPAK     17499   
1     A2AAJ9         0  WSKGGLELQGDTRREARQQGCVAELLLRDLRREDAGEYSCT      4886   
2     Q02817         0  QCHCRLHGHLYTPGQEITNDCEQCVCNAGRWVCKDLPCPGT       370   
3     A2ASS6         0  MDVTRSSVFLSWTKPIYDGGCEIQGYIVEKCDVSVGEWTMC     24656   
4     A2ASS6         0  ESQLERLYKPVFVLKPASFKCLEGQTARFDLKVVGRPMPET      1474   
...      ...       ...               

In [6]:
print(top_ps.groupby("id").size())

id
A2AAJ9    237
A2ASS6    514
Q02817    215
Q91ZX7    332
Q9Y6R7    435
dtype: int64


In [7]:
df = df_merged

print(df.shape)

tm = get_top_n_proteins(df, 20, "id")
print(tm)

(22316, 4)
number of unique proteins in final df: 20
          id  modified                                   sequence  position  \
0     P00533         0  QKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSC       506   
1     P21817         0  ERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPK       936   
2     P98156         0  IQCGSGECIHKKWRCDGDPDCKDGSDEVNCPSRTCRPDQFE       264   
3     A2ASS6         0  SEPSDPAVAGDPIYPPGPPSCPEVKDKTKSSISLAWKPPAK     17499   
4     O00468         0  RSCRVNPRTRRPEMLLRPESCPARQAPVCGDDGVTYENDCV       349   
...      ...       ...                                        ...       ...   
3268  Q7TMY8         0  TKETNCESDRERGSKQACSPCSSQSSSSGICTDFWDLLVKL      3376   
3269  Q7TMY8         0  IGEILIQGLTEDMVTVLIRACVSMLGVPVDPDTLHATLRLC      1762   
3270  Q7TMY8         0  TRDTVLKLLLNGARHLGYTLCKQIGTLLAELREYNLEQQRR      3636   
3271  P19137         0  KNCDRCKPGFYNLKERNPEGCSECFCFGVSGVCDSLTWSIS       507   
3272  P19137         0  DTGLGCVPCNCSVEGSVSDNCTEEGQCHCGPGVSGKQCDRC       970   

In [8]:
def create_protein_summary_table(df, colname):
    table = df.groupby(colname)["position"].agg(['mean', 'median', 'std'])
    table = table.reset_index()
    
    return table

tab = create_protein_summary_table(df, "id")
print(tab)

          id          mean   median           std
0     A2A432        447.75    351.5    298.692339
1     A2AAJ9    4084.78903   4095.0   2388.087334
2     A2AJI0         324.0    369.0    136.460983
3     A2AN08   2733.631148   2689.5   1417.072195
4     A2ASS6  16180.031128  15353.5  10127.952696
...      ...           ...      ...           ...
1567  Q9Z2W0         240.7    253.0    127.468558
1568  Q9Z2X1    189.166667    194.5    153.107043
1569  Q9Z2Z6         124.0    112.5     91.834634
1570  Q9ZNZ7        857.64    783.0     508.61838
1571  Q9ZVA2         379.0    369.5      37.12632

[1572 rows x 4 columns]


In [9]:
#print(df.head())

df["mean_position"] = df.groupby("id")["position"].transform(func = "mean")
print(df)

df[df["id"] == "Q14584"][["id", "position", "mean_position"]]

           id  modified                                   sequence  position  \
0      Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1      Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2      P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3      Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4      P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   
...       ...       ...                                        ...       ...   
22311  P19137         0  KNCDRCKPGFYNLKERNPEGCSECFCFGVSGVCDSLTWSIS       507   
22312  Q62059         0  DQCELDFDECHSNPCRNGATCVDGFNTFRCLCLPSYVGALC      3104   
22313  P19137         0  DTGLGCVPCNCSVEGSVSDNCTEEGQCHCGPGVSGKQCDRC       970   
22314  Q8C8R3         0  SGVESTQADLPNESSSLSSRCAIPEGNESAKEIASPSSPVK      2810   
22315  O88703         0  GEICLLTRGRRTASVRADTYCRLYSLSVDNFNEVLEEYPMM       601   

       mean_position  
0               

,id,position,mean_position
1,Q14584,441,313.305556
126,Q14584,150,313.305556
181,Q14584,385,313.305556
1251,Q14584,301,313.305556
1458,Q14584,494,313.305556
1764,Q14584,108,313.305556
1832,Q14584,326,313.305556
3162,Q14584,146,313.305556
3308,Q14584,497,313.305556
3806,Q14584,270,313.305556


In [10]:
print(df_merged.head())
print(tab.head())
def df_to_csv(df, file_path, index_flag):
    
    df.to_csv(f"{file_path}/tab2.csv", index = index_flag)
    
    print(f"{df} saved in {file_path}")

path = "/home/callum/projects/sno_analysis/data/processed"

df_to_csv(tab, path, True)


       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   

   mean_position  
0           81.4  
1     313.305556  
2          278.4  
3    1381.157895  
4          162.0  
       id          mean   median           std
0  A2A432        447.75    351.5    298.692339
1  A2AAJ9    4084.78903   4095.0   2388.087334
2  A2AJI0         324.0    369.0    136.460983
3  A2AN08   2733.631148   2689.5   1417.072195
4  A2ASS6  16180.031128  15353.5  10127.952696


          id          mean   median           std
0     A2A432        447.75    351.5    298.692339
1     A2AAJ9    4084.78903   4095.0   2388.087334
2     A2AJI0         324.0    369.0    136.460983
3     A2AN08   2733.631148   2689.5   1417.072195
4     A2ASS6  16180.031128  15353.5  10127.952696
...      ...           ...      ...           ...
1567  Q9Z2W0         240.7    253.0    127.468558
1568  Q9Z2X1    189.166667    194.5    153.107043
1569  Q9Z2Z6         124.0    112.5     91.834634
1570  Q9ZNZ7        857.64    783.0     508.61838
1571  Q9ZVA2         379.0    369.5      37.12632

[1572 rows x 4 columns] saved in /home/callum/projects/sno_analysis/data/processed


In [11]:
def get_highly_modified_proteins(df, id_col_name, threshold_number):
    series = df.groupby([id_col_name]).size()

    #filter based on x modifications 
    mask = series >= threshold_number
    excluded_mask = series < threshold_number
    # apply the mask
    series_filtered = series[mask]
    series_filtered = series_filtered.reset_index()

    excluded = series[excluded_mask]
    excluded = excluded.reset_index()

    # # check how many proteins appear x or more times

    print(f"proteins > {threshold_number}:{series_filtered.shape}")

    # merge using array of protein ids 

    df_merged = pd.merge(df, series_filtered, on = "id", how = "inner", indicator = False)

    excluded_rows = pd.merge(df, excluded, on="id", how="inner")

    print("================##===============")
    print(f"proteins > {threshold_number}:{series_filtered.shape}")
    print(f"original df:{df.shape}")
    print(f"new dataframe: {df_merged.shape}")
    print(f"ecluded: {excluded.shape}")
    print("================##===============")

    # sanity check

    print(f"{df_merged.shape[0] + excluded_rows.shape[0] == df.shape[0]}")
    
    return df_merged





In [12]:
import pandas as pd
import numpy as np
file_path = "/home/callum/projects/sno_analysis/data/processed"


df_log = pd.read_csv(f"{file_path}/highly_modified.csv")

df_log["log(occurrences)"] = np.log(df_log["occurrences"])

def add_log_column(df, path, colname):
    
    df_log = pd.read_csv(f"{path}/{df}")
    
    df_log[f"log_{colname}"] = np.log(df_log[f"{colname}"])


In [13]:
file_path = "/home/callum/projects/sno_analysis/data/processed"

df_clean = pd.read_csv(f"{file_path}/df_clean.csv")


# Final product: a new column on df_clean that buckets each row's position value into categorical ranges — 
# e.g. "0–500", "500–1000", "1000+", or however many bins you choose. Package/method hint: pd.cut() — 
# look up its signature, specifically the bins argument (you can pass either a number of equal-width bins, 
# or explicit boundary edges) and the optional labels argument for naming the resulting categories something readable 
# instead of raw interval notation.

df_clean["position_bin"] = pd.cut(df_clean["position"], bins = [0, 500, 1000, np.inf], labels = ["0-500", "501-1000", "1000+"])
print(df_clean.head())


#sanity check


print(df_clean["position_bin"].value_counts().sum())
print(df_clean.shape[0])



       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   

   mean_position  sequence_length centre_residue position_bin  
0      81.400000               41              C        0-500  
1     313.305556               41              C        0-500  
2     278.400000               41              C        0-500  
3    1381.157895               41              C        1000+  
4     162.000000               41              C        0-500  
24061
24061


In [14]:
# 3. Boolean flag with .isin()
# Final product: a new column on df_clean, True/False, 
# flagging whether each row's id belongs to some list of interest — the natural choice here is your top-N proteins list from earlier. 
# Method hint: df["id"].isin(some_list_of_ids) returns a boolean Series directly, ready to assign as a new column.

#load in df
top20 = pd.read_csv(f"{file_path}/top20_proteins.csv")
print(top20.head())
# getting list of ids to supply
series = top20["id"] 

df_clean["interesting"] = df_clean["id"].isin(series)

print(df_clean.head())

print(df_clean.groupby(df_clean["interesting"]).size())



       id  modified                                   sequence  position  \
0  P00533         0  QKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSC       506   
1  P21817         0  ERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPK       936   
2  P98156         0  IQCGSGECIHKKWRCDGDPDCKDGSDEVNCPSRTCRPDQFE       264   
3  A2ASS6         0  SEPSDPAVAGDPIYPPGPPSCPEVKDKTKSSISLAWKPPAK     17499   
4  O00468         0  RSCRVNPRTRRPEMLLRPESCPARQAPVCGDDGVTYENDCV       349   

   occurrences  
0           60  
1          100  
2           67  
3          514  
4          143  
       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRN

In [15]:
# 1. Sort by protein, then position

# Objective: order your row-level data (df_clean) so it reads naturally — grouped by protein, and within each protein, 
# sites in position order. 
# Package/method hint: .sort_values(), which accepts a list of column names when you want multi-level sorting
# (sort by the first column, then break ties using the second). Deliverable: 
# a DataFrame with the same rows and columns as df_clean, just reordered.

print(df_clean.columns
      )
df_clean_sorted = df_clean.sort_values(by=["id", "position"])

print(df_clean.head())
print(df_clean_sorted.head())




Index(['id', 'modified', 'sequence', 'position', 'mean_position',
       'sequence_length', 'centre_residue', 'position_bin', 'interesting'],
      dtype='str')
       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   

   mean_position  sequence_length centre_residue position_bin  interesting  
0      81.400000               41              C        0-500        False  
1     313.305556               41              C        0-500        False  
2     278.400000               41              C        0-500        False  
3    1381.157895               41              C        1000+        False

In [16]:
# 2. Rank sites within a protein

# Objective: for each protein, assign a rank to its sites based on position 
# (e.g. "this is the 1st, 2nd, 3rd... site on this protein, in position order"). 
# This is conceptually similar to .transform() from a couple of sessions ago — you need something that operates
# within each group and returns a value per row, not collapsed down to one row per group. Method hint: 
# groupby("id")["position"].rank() — look up what rank() actually does (it's similar to sorting, but assigns a 
# numeric rank rather than reordering rows), and check its method= argument, which matters if two sites in the same 
# protein ever share the exact same position (unlikely here, but worth knowing what pandas does by default if it happened — 
# "first," "min," "average" are common options).

df_clean_sorted["position_rank"] = df_clean_sorted.groupby("id")["position"].rank(method = "average").astype("int")

print(df_clean_sorted.head(30))

           id  modified                                   sequence  position  \
7047   A2A432         0  ENDTDSEDNSSETSNQERRRCRQGPPRPPYPPLLPPVFPPP        32   
10710  A2A432         0  PPPQVRRTRGLQDLGAMKSVCPGTSGFSSPNPSAASAAAQE        74   
12208  A2A432         0  SSSSTPPQPRDSASPSTSSFCLGVPVATSSHVPIQKKLRFE       156   
3789   A2A432         0  QNSTSIKYNLEELYQAVENLCSHKISANLYKQLRQICEDHI       305   
4153   A2A432         0  VENLCSHKISANLYKQLRQICEDHIKAQIHQFREDSLDSVL       321   
631    A2A432         0  IHQFREDSLDSVLFLKKIDRCWQNHCRQMIMIRSIFLFLDR       349   
13691  A2A432         0  EDSLDSVLFLKKIDRCWQNHCRQMIMIRSIFLFLDRTYVLQ       354   
4446   A2A432         0  KTMVQELLDFKDKVDHIIDTCFLKNEKFINAMKEAFETFIN       590   
11279  A2A432         1  VGKSASVDAEKSMLSKLKHECGAAFTSKLEGMFKDMELSKD       694   
13744  A2A432         0  TFYLGKHSGRKLQWQSTLGHCVLKAEFKEGKKELQVSLFQT       787   
3155   A2A432         0  IKHATGIEDGELRRTLQSLACGKARVLAKNPKGKDIEDGDK       844   
8455   A2A432         0  ARVLAKNPKGKDIED

In [17]:
# 3. Most clustered vs. most spread-out proteins

# Objective: using your existing protein summary table (mean/median/std of position per protein — already built),
# identify which proteins have the tightest clustering of sites (low std) versus the most spread out (high std). 
# No new computation needed — this is purely a sorting/selection task on data you already have. Method hint: 
#     .sort_values("std") ascending for most clustered, descending (or .nlargest(), which you've used before) for most spread out.
file_dir = "/home/callum/projects/sno_analysis/data/processed"
df_name = "protein_summary.csv"
summary_df = pd.read_csv(f"{file_dir}/{df_name}")
print(summary_df.head())


# deliver top 10 and lowest 10 clustering of sites 
summary_sorted = summary_df.sort_values("std", ascending=True)
summary_sorted_top10 = summary_sorted.head(10)
print(summary_sorted_top10)



#broken one
def get_ranked_std_summary(summary_df, highest_or_lowest, file_path):
    if highest_or_lowest == "highest":
        name = "top"
        ascending_flag = False
    elif highest_or_lowest =="lowest":
        name = "bottom"
        ascending_flag = True
    else:
        print("please choose either highest or lowest")
        
    summary_sorted = pd.read_csv(summary_df)
    
    summary_sorted = summary_df.sort_values("std", ascending=ascending_flag)
    
    pd.to_csv(f"{file_path}/{summary_sorted}_{name}.csv")
    
    print(f"new file saved at{file_path}/{summary_sorted}_{name}.csv")

#corrected one 


file_path = "/home/callum/projects/sno_analysis/data/processed"
file = f"{file_path}/protein_summary.csv"

summary_df = pd.read_csv(file)

def get_ranked_std_summary(summary_df, highest_or_lowest, file_path, n):
    if highest_or_lowest == "highest":
        name = "top"
        ascending_flag = False
    elif highest_or_lowest =="lowest":
        name = "bottom"
        ascending_flag = True
    else:
        print("please choose either highest or lowest")
        return

    summary_sorted = summary_df.sort_values("std", ascending=ascending_flag)
    summary_sorted = summary_sorted.head(n)

    output_path = f"{file_path}/protein_std_{name}_top{n}.csv"
    summary_sorted.to_csv(output_path, index=False)

    
    print(f"new file saved at{file_path}/{summary_sorted}_{name}.csv")




get_ranked_std_summary(summary_df, "lowest", file_path, 10)







       id         mean  median          std
0  A2A432   447.750000   351.5   298.692339
1  A2AAJ9  4084.789030  4095.0  2388.087334
2  A2AJB2    65.000000    83.0    33.808283
3  A2AJI0   324.000000   369.0   136.460983
4  A2AN08  2733.631148  2689.5  1417.072195
          id   mean  median       std
2043  Q9NR33   84.5    84.5  0.707107
1134  Q3ZCI9  148.5   148.5  0.707107
838   P62897   16.5    16.5  2.121320
1152  Q5MK14   29.5    29.5  2.121320
1978  Q9FX54  158.0   158.0  2.828427
840   P62911   93.5    93.5  3.535534
528   P31081  444.5   444.5  3.535534
858   P63039  444.5   444.5  3.535534
778   P61014   41.0    41.0  5.000000
1149  Q5HZV9  108.0   108.0  5.656854
new file saved at/home/callum/projects/sno_analysis/data/processed/          id   mean  median       std
2043  Q9NR33   84.5    84.5  0.707107
1134  Q3ZCI9  148.5   148.5  0.707107
838   P62897   16.5    16.5  2.121320
1152  Q5MK14   29.5    29.5  2.121320
1978  Q9FX54  158.0   158.0  2.828427
840   P62911   93.5    

In [18]:
df = pd.read_csv(file)
df.dtypes

id            str
mean      float64
median    float64
std       float64
dtype: object

In [19]:
# add sequence legnth column - these should all be the same 
df_clean["sequence_length"] = df_clean["sequence"].str.len()

print(df_clean.head())

print(df_clean["sequence_length"].nunique())

       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   

   mean_position  sequence_length centre_residue position_bin  interesting  
0      81.400000               41              C        0-500        False  
1     313.305556               41              C        0-500        False  
2     278.400000               41              C        0-500        False  
3    1381.157895               41              C        1000+        False  
4     162.000000               41              C        0-500        False  
1


In [20]:
# find centre residue from each sequence - these should always be C as the sequence is centred around the modified residue
# mid_value = df_clean["sequence"].str.len() // 2
# print(mid_value)

# we know the legnths are all the same
seq_len = df_clean["sequence_length"].iloc[0]
#get midpoint value from length
midpoint = seq_len // 2
#apply this value to index the sequence to get the residue
df_clean["centre_residue"] = df_clean["sequence"].str[midpoint]
print(df_clean.head())
print(df_clean["centre_residue"].nunique())

       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   

   mean_position  sequence_length centre_residue position_bin  interesting  
0      81.400000               41              C        0-500        False  
1     313.305556               41              C        0-500        False  
2     278.400000               41              C        0-500        False  
3    1381.157895               41              C        1000+        False  
4     162.000000               41              C        0-500        False  
1


The standard pattern to avoid it, going forward:

Keep three separate, clearly-named things, and never let them blur together:


# 1. The folder — a string, defined once, reused everywhere
processed_dir = "/home/callum/projects/sno_analysis/data/processed"

# 2. The filename — a string, built from other strings/values, NEVER from a DataFrame
filename = f"protein_summary_{name}.csv"

# 3. The full path — combining 1 and 2, still just a string
output_path = f"{processed_dir}/{filename}"

# THEN, and only then, call .to_csv() on the actual DataFrame, passing the path as its argument
summary_sorted.to_csv(output_path, index=False)



The naming discipline that makes this safe: give path-related variables names that make their "stringiness" obvious — _path, _dir, _filename suffixes are a good habit (you can see this already forming in things like data_path, file_path from earlier). Never let a DataFrame variable share a similar-sounding name to a path variable in the same block of code — that's part of how the confusion crept in.

One more robustness upgrade worth knowing, since you'll hit this repeatedly in Project 2 and beyond: os.path.join() or pathlib.Path are safer than f-string concatenation for building paths, because they handle slashes correctly regardless of OS or whether a variable already has a trailing slash:

python
from pathlib import Path
output_path = Path(processed_dir) / filename

Not essential to switch immediately, but worth knowing it exists — f-string concatenation works fine as long as you're careful, but it's easy to accidentally produce data/processed//file.csv (double slash) if you're not consistent about trailing slashes.

In [21]:
import pandas as pd
import numpy as np
file_path = "/home/callum/projects/sno_analysis/data/processed"
df = pd.read_csv(f"{file_path}/df_clean.csv") 

print(df.columns)

mod = df["modified"].to_numpy()
print(np.unique(mod, return_counts = True))

print("Unique protein IDs:", df["id"].nunique(), f"\ntotal proteins:", df.shape[0])

id_list = df["id"].tolist()

df = df.head(11)
print(df)

Index(['id', 'modified', 'sequence', 'position', 'mean_position',
       'sequence_length', 'centre_residue'],
      dtype='str')
(array([0, 1]), array([20287,  3774]))
Unique protein IDs: 2217 
total proteins: 24061
        id  modified                                   sequence  position  \
0   Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1   Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2   P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3   Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4   P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   
5   Q9NYY3         1  TITYQPAASTKMCEQALGKGCGADSKKKRPPQPPEESQPPQ        26   
6   Q9Z1G4         0  AEEFDFGDTMVHQAIHTIEYCLGCISNTASYLRLWALSLAH       730   
7   Q9BXJ9         0  EGTSAVENLNEMQCMWFQTECAQAYKAMNKFGEALKKCHEI       491   
8   Q91VD9         0  KSKKAREGVMEFLLANHPLDCPICDQGGECDLQDQSMMFGS       128   
9   P07814   

In [22]:
items = list(range(10))  # [0, 1, 2, ..., 9]
batch_size = 3


# [0, 1, 2]
# [3, 4, 5]
# [6, 7, 8]
# [9]

In [31]:
KYTE_DOOLITTLE = {
    "A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5,
    "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
    "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6,
    "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2
}


CHARGE = {
    "D": -1, "E": -1,   # acidic, negatively charged
    "K": 1, "R": 1,     # basic, positively charged
    "H": 0,             # often treated as neutral/weakly basic; some schemes give H = +1 — worth a footnote on which convention you picked
    # everything else defaults to 0
}





# add charges 
def get_charge_sum(seq, window = 5):
    
    cent = int(len(seq) // 2)    
    if seq[cent] == "C":  
        charges = []
        charges.append(0.0)
        start = -(window)
        end = window + 1
        for i in range(start, end):
            if i != 0:
                if seq[cent + i] in list(CHARGE.keys()):
                    charges.append(CHARGE[seq[cent + i]])
    
    return np.sum(charges)

df["charge_10"] = df["sequence"].apply(lambda seq: get_charge_sum(seq, window=10))




# add hydrphobicity 


def get_hydrophobicity_sum(seq, window = 5):
    
    cent = int(len(seq) // 2)    
    if seq[cent] == "C":  
        hphobics = []
        start = -(window)
        end = window + 1
        for i in range(start, end):
            if seq[cent + i] in ["-", "U"]:
                continue
            if i != 0:
                hphobics.append(KYTE_DOOLITTLE[seq[cent + i]])
    
    return np.sum(hphobics)




df["hydrophobicity_5"] = df["sequence"].apply(lambda seq: get_hydrophobicity_sum(seq, window=5)
)  
df["hydrophobicity_10"] = df["sequence"].apply(lambda seq: get_hydrophobicity_sum(seq, window=10)
)  
df["charge_5"] = df["sequence"].apply(lambda seq: get_charge_sum(seq, window=5)
)  
df["charge_10"] = df["sequence"].apply(lambda seq: get_charge_sum(seq, window=10)
)  




In [25]:


# df = df.merge(
#     df_clean[["id", "position", "sequence"]],
#     on=["id", "position"],
#     how="left"
# )

print(df.head())

   Unnamed: 0      id  modified  position  hydrophobicity_5  \
0           0  Q9Y2Z0         1        88             -15.9   
1           1  Q14584         0       441               1.7   
2           2  P00390         0       328              13.1   
3           3  Q5UIP0         0      1735             -14.9   
4           4  P0C8J8         0        17               8.8   

   hydrophobicity_10  charge_5  charge_10  cluster_id  \
0              -16.6       1.0        1.0       266.0   
1              -10.2      -1.0       -1.0       180.0   
2               13.0      -2.0       -1.0       193.0   
3              -38.3       0.0        3.0        17.0   
4                6.3      -1.0        1.0       235.0   

                                    sequence  
0  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL  
1  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK  
2  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG  
3  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN  
4  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRN

In [24]:
df = df.drop(columns=["sequence_x"])
df = df.rename(columns={"sequence_y": "sequence"})


# save as new file

pre_split = pre_split[["id", "modified", "position", "hydrophobicity_5", "hydrophobicity_10", "charge_5", "charge_10"]]
print(pre_split.head())

pre_split.to_csv("/home/callum/projects/sno_analysis/data/processed/pre_split.csv")


       id  modified  position  hydrophobicity_5  hydrophobicity_10  charge_5  \
0  Q9Y2Z0         1        88             -15.9              -16.6       1.0   
1  Q14584         0       441               1.7              -10.2      -1.0   
2  P00390         0       328              13.1               13.0      -2.0   
3  Q5UIP0         0      1735             -14.9              -38.3       0.0   
4  P0C8J8         0        17               8.8                6.3      -1.0   

   charge_10  
0        1.0  
1       -1.0  
2       -1.0  
3        3.0  
4        1.0  


In [ ]:
# import requests
# import os 

# # Source - https://stackoverflow.com/a/89243
# # Posted by David Cournapeau, modified by community. See post 'Timeline' for change history
# # Retrieved 2026-09-17, License - CC BY-SA 4.0

# import subprocess




# file_path = "/home/callum/projects/sno_analysis/data/homology/int"
# out_path = "/home/callum/projects/sno_analysis/data/homology/int"
# batch = 50
# with open(f"{file_path}/cd.fasta", "w") as f:
#     for i in range(0, len(ids), batch):
#         chunk = ids[i : i + batch]
        
#         for uniprot_id in chunk:
#             url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
#             response = requests.get(url)

#             if response.status_code == 200:
#                 fasta = response.text
#                 print(fasta)
#                 for line in fasta.splitlines():
#                     if line.startswith(">"):
#                         # f.write(f"\n")
#                         parts = line.split("|")
#                         id = parts[1]
#                         f.write(f">{id}\n")
#                         continue
                        
#                     f.write(f"{line}\n")
#             else:
#                 print(f"Failed to retrieve {uniprot_id}")


# code = "cd-hit -i /home/callum/projects/sno_analysis/data/homology/int/cd.fasta -o /home/callum/projects/sno_analysis/data/homology/int/test -c 0.4 -n 2 -d 0"
# code = code.split(" ")
# subprocess.run(code) 


>sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens OX=9606 GN=TP53 PE=1 SV=4
MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP
DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAK
SVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHE
RCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNS
SCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELP
PGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPG
GSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD

>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha OS=Homo sapiens OX=9606 GN=HBA1 PE=1 SV=2
MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG
KKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTP
AVHASLDKFLASVSTVLTSKYR

>sp|P68871|HBB_HUMAN Hemoglobin subunit beta OS=Homo sapiens OX=9606 GN=HBB PE=1 SV=2
MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPK
VKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFG
KEFTPPVQAAYQKVVAGVANALAHKYH

Program: CD-HIT, V4.8.1 (+OpenMP), Apr

CompletedProcess(args=['cd-hit', '-i', '/home/callum/projects/sno_analysis/data/homology/int/cd.fasta', '-o', '/home/callum/projects/sno_analysis/data/homology/int/test', '-c', '0.4', '-n', '2', '-d', '0'], returncode=0)

In [ ]:
# get ids from returned fasta file
# return list of ids
# this replaces list moving forward

       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   

   mean_position  sequence_length centre_residue  
0      81.400000               41              C  
1     313.305556               41              C  
2     278.400000               41              C  
3    1381.157895               41              C  
4     162.000000               41              C  
(500, 7)


In [1]:
import subprocess


In [6]:
import subprocess
import requests
from tqdm import tqdm
import os 
import pandas as pd

pre_split = pd.read_csv("/home/callum/projects/sno_analysis/data/processed/pre_split.csv")

# create id list from pre_split data 
ids = pre_split["id"]

batch = 50
def fetch_uniprot_sequences(ids, batch):
    missing = []
    id_dict = {}
    for i in tqdm(range(0, len(ids), batch)):
        
        chunk = ids[i : i + batch]
        #print(chunk)
        
        for uniprot_id in chunk:
            url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
            response = requests.get(url)
            
            #print(response.text)

            if response.status_code == 200:
                fasta = response.text
                #print(fasta)
                for line in fasta.splitlines():
                    if line.startswith(">"):
                        # f.write(f"\n")
                        parts = line.split("|")
                        
                        id = parts[1]
                        id_dict[id] = ""
                        #print(id)
                        #f.write(f">{id}\n")
                        continue
                        
                    # f.write(f"{line}\n")
                    id_dict[id] += line
            else:
                print(f"Failed to retrieve {uniprot_id}")
                missing.append(uniprot_id)
    return id_dict, missing

dic, missing = fetch_uniprot_sequences(ids, 2)

print(dic)

file_path = "/home/callum/projects/sno_analysis/data/processed/test"
out_path = "/home/callum/projects/sno_analysis/data/homology/int"
def create_fasta(dic):
    with open(f"{file_path}/Test.fasta", "w") as f:
        for id, seq in dic.items():
            f.write(f">{id}\n{seq}\n")
            
    f.close()

create_fasta(dic)


code = "cd-hit -i /home/callum/projects/sno_analysis/data/processed/test/Test.fasta -o /home/callum/projects/sno_analysis/data/homology/int/test.txt -c 0.4 -n 2 -d 0"
code = code.split(" ")
subprocess.run(code) 



  0%|          | 0/250 [00:00<?, ?it/s]

100%|██████████| 250/250 [05:01<00:00,  1.21s/it]


{'Q9Y2Z0': 'MAAAAAGTATSQRFFQSFSDALIDEDPQAALEELTKALEQKPDDAQYYCQRAYCHILLGNYCVAVADAKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKLDIETGFHRVGQAGLQLLTSSDPPALDSQSAGITGADANFSVWIKRCQEAQNGSESEVWTHQSKIKYDWYQTESQVVITLMIKNVQKNDVNVEFSEKELSALVKLPSGEDYNLKLELLHPIIPEQSTFKVLSTKIEIKLKKPEAVRWEKLEGQGDVPTPKQFVADVKNLYPSSSPYTRNWDKLVGEIKEEEKNEKLEGDAALNRLFQQIYSDGSDEVKRAMNKSFMESGGTVLSTNWSDVGKRKVEINPPDDMEWKKY', 'Q14584': 'MLENYKNLATVGYQLFKPSLISWLEQEESRTVQRGDFQASEWKVQLKTKELALQQDVLGEPTSSGIQMIGSHNGGEVSDVKQCGDVSSEHSCLKTHVRTQNSENTFECYLYGVDFLTLHKKTSTGEQRSVFSQCGKAFSLNPDVVCQRTCTGEKAFDCSDSGKSFINHSHLQGHLRTHNGESLHEWKECGRGFIHSTDLAVRIQTHRSEKPYKCKECGKGFRYSAYLNIHMGTHTGDNPYECKECGKAFTRSCQLTQHRKTHTGEKPYKCKDCGRAFTVSSCLSQHMKIHVGEKPYECKECGIAFTRSSQLTEHLKTHTAKDPFECKICGKSFRNSSCLSDHFRIHTGIKPYKCKDCGKAFTQNSDLTKHARTHSGERPYECKECGKAFARSSRLSEHTRTHTGEKPFECVKCGKAFAISSNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAKKPFTCMECGKAFKFPTCVNLHMRIHTGEKPYKCKQCGKSFSYSNSFQLHERTHTGEKPYECKECGKAFSSSSSFRNHERRHADERLSA', 'P00390': 'MALLPRALSAGAGPSWRRAARAFRGFLLLLPEPAALTRALSRAMAC

CompletedProcess(args=['cd-hit', '-i', '/home/callum/projects/sno_analysis/data/processed/test/Test.fasta', '-o', '/home/callum/projects/sno_analysis/data/homology/int/test.txt', '-c', '0.4', '-n', '2', '-d', '0'], returncode=0)

In [ ]:
# # get ids from returned fasta file
# # return list of ids
# # this replaces list moving forward
# import os 
# out_path = "/home/callum/projects/sno_analysis/data/homology/int"
# file_name = "test.txt"
# cleaned_list = "cleaned_ids.txt"
# file_path = "/home/callum/projects/sno_analysis/data/processed/test"

# clusters = {}


# with open(os.path.join(out_path, file_name), "r") as f:
#     cleaned_list = []
#     for l in f:
#         if l.startswith(">"):
#             l = l.strip(">")
#             cleaned_list.append(l.strip())
            
# print(cleaned_list)

['P04637', 'P68871']


In [ ]:
# # alternative, and better method - preserve all sequences, but cluster together ones highlighted in the cd-hit pipeline

# cluster_path = "/home/callum/projects/sno_analysis/data/homology/int/test.txt.clstr"

# with open(cluster_path) as f:
#     for line in f:
#         if line.startswith(">"):
#             id = line.strip(">").split(" ")
#             id = f"cluster_{id[1]}"

# with open(cluster_path) as f:
#     tag = ", >"
#     for line in f:
#         if line.startswith(">"):
#             continue
#         if tag in line:
#             parts = line.split(" ")
#             id2 = parts[1].strip(">...")
#             print(id2)

P04637
P69905
P68871


In [7]:
# got chat to write the rest to save time 

cluster_path = "/home/callum/projects/sno_analysis/data/homology/int/test.txt.clstr"

clusters = {}

with open(cluster_path) as f:
    for line in f:
        line = line.strip()

        if line.startswith(">Cluster"):
            cluster_id = line.split()[1]
            clusters[cluster_id] = []

        elif ", >" in line:
            protein_id = line.split(", >")[1].split("...")[0]
            clusters[cluster_id].append(protein_id)
            
print(clusters)


# reverse dict

protein_to_cluster = {
    protein_id: cluster_id
    for cluster_id, protein_ids in clusters.items()
    for protein_id in protein_ids
}


pre_split["cluster_id"] = pre_split["id"].map(protein_to_cluster)

{'0': ['A2ASS6'], '1': ['A2AAJ9'], '2': ['Q09666'], '3': ['Q9Y6R7'], '4': ['Q02817'], '5': ['A2AN08'], '6': ['P21817'], '7': ['Q9JHU4'], '8': ['P78527'], '9': ['Q62407'], '10': ['P48725'], '11': ['Q9QYR6'], '12': ['Q8VHX6', 'P21333', 'Q8BTM8', 'O75369', 'Q80X90'], '13': ['Q92616'], '14': ['P70398'], '15': ['Q9Y490'], '16': ['P19096', 'P49327'], '17': ['Q5UIP0'], '18': ['P14873'], '19': ['Q9UGM3'], '20': ['P31327', 'B2RQC6'], '21': ['Q9ESD7'], '22': ['O00468'], '23': ['Q14008'], '24': ['P35580', 'Q02566'], '25': ['Q6PDN3'], '26': ['Q8CC88'], '27': ['Q6NS46'], '28': ['Q5SSW2'], '29': ['Q9ER60'], '30': ['Q6ZPY7'], '31': ['Q68FD5', 'P11442'], '32': ['P46940'], '33': ['Q9ZNZ7'], '34': ['Q9P2E9', 'Q99PL5'], '35': ['Q09429'], '36': ['P11388'], '37': ['P07814', 'Q8CGC7'], '38': ['Q12879'], '39': ['Q6PAR5'], '40': ['P42704', 'Q6PB66'], '41': ['O54991'], '42': ['P06213'], '43': ['P35918'], '44': ['Q7TPV4', 'Q9BQG0'], '45': ['Q9JIX8'], '46': ['Q00519'], '47': ['Q6A026'], '48': ['P70388'], '49': [

In [15]:
new = pre_split.iloc[:, 1:]
print(new.shape)
print(new.head())


new.to_csv("/home/callum/projects/sno_analysis/data/processed/most_features.csv")

(500, 8)
       id  modified  position  hydrophobicity_5  hydrophobicity_10  charge_5  \
0  Q9Y2Z0         1        88             -15.9              -16.6       1.0   
1  Q14584         0       441               1.7              -10.2      -1.0   
2  P00390         0       328              13.1               13.0      -2.0   
3  Q5UIP0         0      1735             -14.9              -38.3       0.0   
4  P0C8J8         0        17               8.8                6.3      -1.0   

   charge_10 cluster_id  
0        1.0        266  
1       -1.0        180  
2       -1.0        193  
3        3.0         17  
4        1.0        235  


In [ ]:
# save new pre_split with cluster ids added


print(len(pre_split["cluster_id"]))
print(pre_split["cluster_id"].nunique())

24061
361


>Cluster 0

0	393aa, >P04637... *

>Cluster 1

0	142aa, >P69905... at 45.77%

1	147aa, >P68871... *



In [16]:
from collections import Counter 
import pandas as pd
import numpy as np
file_path = "/home/callum/projects/sno_analysis/data/processed"


print(df.head())
seq1 = "AKGTL"
seq2 = "SKTTG"
seq3 = "PLRTR"
lst = [seq1, seq2, seq3]
def get_pos_frequency_matrix(seq_list):
    # no_sequences = len(seq_list)
    pos_dict = {}
    for position in range(0, len(seq_list[0])):
        res_in_this_position = []
        for seq in seq_list:
            res_in_this_position.append(seq[position])
        pos_dict[position + 1] = Counter(res_in_this_position)
        
    return pos_dict
pos_dict = get_pos_frequency_matrix(lst)
print(pos_dict[2])

# amino acid letter
aas = ['A', 'R', 'N', 'D', 'C', 'E', 'Q', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V', 'X']


#determine how many columns to add - the sequence length
seq_length = len(df["sequence"][0])

col_names = np.arange(1, seq_length + 1)

# Create empty matrix
matrix = pd.DataFrame(
    index=aas,
    columns=col_names
)
matrix = matrix.fillna(0) 
print(matrix)


NameError: name 'df' is not defined

In [ ]:
# file_path = "/home/callum/projects/sno_analysis/data/processed"

# df = pd.read_csv("/home/callum/projects/sno_analysis/data/processed/most_features.csv")

# amino acid letter
aas = ['A', 'R', 'N', 'D', 'C', 'E', 'Q', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V', 'X']


#determine how many columns to add - the sequence length
seq_length = len(pre_split["sequence"][0])

col_names = np.arange(1, seq_length + 1)

# Create empty matrix
matrix = pd.DataFrame(
    index=aas,
    columns=col_names
)
matrix = matrix.fillna(0) 
print(matrix)


  1  2  3  4  5  6  7  8  9  10  ... 31 32 33 34 35 36 37 38 39 40
A  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
R  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
N  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
D  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
C  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
E  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
Q  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
G  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
H  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
I  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
L  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
K  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
M  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
F  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  

In [ ]:
# # file_path = "/home/callum/projects/sno_analysis/data/processed"
# # df = pd.read_csv(f"{file_path}/df_clean.csv") 
# # # use cleaned list 
# # mask = df["id"].isin(cleaned_list)
# # df = df[mask]


# # now parse the dict to populate the matrix 


# seqs = df["sequence"]

# def clean_sequences(series):
#     sequences = []
#     for seq in seqs:
#         seq = seq.replace("-", "X")
#         sequences.append(seq)
#     no_sequences = len(sequences)
    
#     return sequences, no_sequences

        
        
# x, y = clean_sequences(seqs)
# print(y)



# def get_pos_weight_matrix(seq_list):
#     pos_dict = {}
#     for position in range(0, 5):
#         res_in_this_position = []
#         for seq in seq_list:
#             res_in_this_position.append(seq[position])
#         pos_dict[position] = Counter(res_in_this_position)
        
#     return pos_dict

# dic = get_pos_frequency_matrix(x)


# def populate_matrix_with_dict(dict, matrix):

#     for i, j in dict.items():
#         for k in j.items():
#             matrix.at[k[0] , i] = k[1]
    
    

#     return matrix



# matrix = populate_matrix_with_dict(dic, matrix)

# def get_position_probability_matrix(matrix, no_sequences):
#     matrix = matrix.to_numpy()
    
#     ppm = matrix/no_sequences
    
#     return ppm

# x = get_position_probability_matrix(matrix, y)
# print(x)

500


NameError: name 'get_pos_frequency_matrix' is not defined

In [47]:
# file_path = "/home/callum/projects/sno_analysis/data/processed"
# df = pd.read_csv(f"{file_path}/df_clean.csv") 
# # use cleaned list 
# mask = df["id"].isin(cleaned_list)
# df = df[mask]


# now parse the dict to populate the matrix 


seqs = df["sequence"]

def clean_sequences(series):
    sequences = []
    for seq in series:
        seq = seq.replace("-", "X")
        sequences.append(seq)
    no_sequences = len(sequences)
    
    return sequences, no_sequences

        
        
x, y = clean_sequences(seqs)




def get_pos_count(seq_list):
    pos_dict = {}
    for position in range(0, len(seq_list[0])):
        res_in_this_position = []
        for seq in seq_list:
            res_in_this_position.append(seq[position])
        pos_dict[position] = Counter(res_in_this_position)
        
    return pos_dict

dic = get_pos_count(x)


def populate_matrix_with_dict(dict, matrix):

    for i, j in dict.items():
        for k in j.items():
            matrix.at[k[0] , i] = k[1]
    
    

    return matrix



matrix = populate_matrix_with_dict(dic, matrix)

def get_position_probability_matrix(matrix, no_sequences):
    matrix = matrix.to_numpy()
    
    matrix = matrix + 0.00001
    
    ppm = matrix/no_sequences
    
    return ppm

x = get_position_probability_matrix(matrix, y)


def get_position_weight_matrix(ppm):
    
    ppm = np.asarray(ppm, dtype=float)

    pwm = np.log2(ppm)
    
    return pwm

y = get_position_weight_matrix(x)



# transpose matrix
y = np.asarray(y, dtype=float)

y = y.T

print(y)
    

[[-3.87832102e+00 -4.64385547e+00 -4.32192752e+00 -4.26534401e+00
  -4.71785601e+00 -4.10780279e+00 -5.50635135e+00 -3.92138973e+00
  -5.38082058e+00 -4.87832059e+00 -3.47393087e+00 -3.92138973e+00
  -4.96578338e+00 -4.71785601e+00 -4.44222170e+00 -3.79585888e+00
  -4.26534401e+00 -6.64385330e+00 -4.96578338e+00 -3.87832102e+00
  -4.57346617e+00]
 [-4.32192752e+00 -4.26534401e+00 -4.57346617e+00 -4.05889321e+00
  -5.15842833e+00 -3.96578383e+00 -4.71785601e+00 -3.87832102e+00
  -5.15842833e+00 -4.32192752e+00 -4.01158751e+00 -4.21089625e+00
  -5.79585768e+00 -4.32192752e+00 -3.96578383e+00 -3.83650086e+00
  -4.44222170e+00 -5.96578248e+00 -5.38082058e+00 -3.75633053e+00
  -4.57346617e+00]
 [-3.64385583e+00 -4.38082118e+00 -5.15842833e+00 -3.64385583e+00
  -4.87832059e+00 -3.96578383e+00 -4.64385547e+00 -3.92138973e+00
  -5.26534346e+00 -4.64385547e+00 -3.96578383e+00 -4.21089625e+00
  -6.15842730e+00 -4.57346617e+00 -3.64385583e+00 -3.96578383e+00
  -4.71785601e+00 -6.15842730e+00 -4.7

In [50]:
mask = df["modified"] == 1
positives = df[mask]
print(positives.head())
print(positives.shape)

    Unnamed: 0      id  modified  position  hydrophobicity_5  \
0            0  Q9Y2Z0         1        88             -15.9   
5            5  Q9NYY3         1        26              -5.9   
11          11  O55137         1        14             -13.8   
13          13  P04222         1       125             -12.6   
19          19  Q6NS46         1      1229              13.2   

    hydrophobicity_10  charge_5  charge_10  cluster_id  \
0               -16.6       1.0        1.0       266.0   
5               -27.9       1.0        4.0       147.0   
11               -8.7      -1.0       -1.0       236.0   
13               -9.8      -1.0        1.0         NaN   
19                5.9       1.0       -1.0        27.0   

                                     sequence  
0   AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL  
5   TITYQPAASTKMCEQALGKGCGADSKKKRPPQPPEESQPPQ  
11  -------MEATLNLEPSGRSCWDEPLSIAVRGLAPEQPVTL  
13  LRGYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQYAYDGK  
19  QVGQAIEATVVDPDVPRAFLCL

essentially i want to create an empty matrix with aa letters in index and number 1-41 as col names. how ca i?

# amino acid letter
aas = ['A', 'R', 'N', 'D', 'C', 'E', 'Q', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

#create matrix skeleton
matrix = pd.DataFrame(list(zip(aas)), columns=['residue'])
matrix = matrix.set_index("residue")
print(df)

#determine how many columns to add - the sequence length
seq_length = len(df["sequence"][0])

col_names = np.arange(1, seq_length + 1)

matrix.columns = col_names
print(matrix)

TypeError: list indices must be integers or slices, not tuple

In [ ]:
# stratisfied k split

print(df.shape)
print(df.head(2))

#print(df["sequence"])

(12, 7)
          id  modified                                   sequence  position  \
595   P04637         0  HSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGT       135   
1096  P68871         1  DGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCV        94   

      mean_position  sequence_length centre_residue  
595           201.9               41              C  
1096          103.5               41              C  


In [ ]:
def compute_pwm_score(window: str, window_offset: int, pwm: np.ndarray, amino_acids: str) -> float:
    score = 0.0
    for local_index, residue in enumerate(window):
        if residue not in amino_acids:
            continue  # e.g. '-' padding — contributes nothing
        pwm_row = window_offset + local_index
        col = amino_acids.index(residue)
        score += pwm[pwm_row, col]
    return score

-1


       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   

   mean_position  sequence_length centre_residue  hydrophobicity_10  \
0      81.400000               41              C              -16.6   
1     313.305556               41              C              -10.2   
2     278.400000               41              C               13.0   
3    1381.157895               41              C              -38.3   
4     162.000000               41              C                6.3   

   hydrophobicity_5  
0             -15.9  
1               1.7  
2              13.1  
3             -14.9  
4     

['D', 'E', 'K', 'R', 'H']


(24061, 11)
       id  modified                                   sequence  position  \
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88   
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441   
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328   
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735   
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17   

   mean_position  sequence_length centre_residue  hydrophobicity_10  \
0      81.400000               41              C              -16.6   
1     313.305556               41              C              -10.2   
2     278.400000               41              C               13.0   
3    1381.157895               41              C              -38.3   
4     162.000000               41              C                6.3   

   hydrophobicity_5  charge_10  charge_5  
0             -15.9        1.0       1.0  
1               1.

       id  modified  position  hydrophobicity_5  hydrophobicity_10  charge_5  \
0  Q9Y2Z0         1        88             -15.9              -16.6       1.0   
1  Q14584         0       441               1.7              -10.2      -1.0   
2  P00390         0       328              13.1               13.0      -2.0   
3  Q5UIP0         0      1735             -14.9              -38.3       0.0   
4  P0C8J8         0        17               8.8                6.3      -1.0   

   charge_10  
0        1.0  
1       -1.0  
2       -1.0  
3        3.0  
4        1.0  


In [ ]:
pre_split

       id  modified  position  hydrophobicity_5  hydrophobicity_10  charge_5  \
0  Q9Y2Z0         1        88             -15.9              -16.6       1.0   
1  Q14584         0       441               1.7              -10.2      -1.0   
2  P00390         0       328              13.1               13.0      -2.0   
3  Q5UIP0         0      1735             -14.9              -38.3       0.0   
4  P0C8J8         0        17               8.8                6.3      -1.0   

   charge_10 cluster_id  
0        1.0        266  
1       -1.0        180  
2       -1.0        193  
3        3.0         17  
4        1.0        235  


In [ ]:
from sklearn.model_selection import StratifiedKFold